# LLM Fine-Tuning в Google Colab

Этот ноутбук позволяет обучить (fine-tune) LLM модель прямо в Google Colab с бесплатным GPU.

**Перед запуском:**
1. `Runtime` → `Change runtime type` → выберите **T4 GPU**
2. Запускайте ячейки по порядку

## 1. Проверка GPU

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 2. Установка зависимостей

In [ ]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes wandb sentencepiece gradio

## 3. Настройка параметров

Здесь можно менять модель, датасет и параметры обучения.

In [ ]:
# === НАСТРОЙКИ ===

# Модель (можно заменить на любую из HuggingFace)
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # маленькая модель для быстрых экспериментов
# MODEL_NAME = "meta-llama/Llama-2-7b-hf"  # для серьёзного обучения (нужен доступ)
# MODEL_NAME = "mistralai/Mistral-7B-v0.1"  # альтернатива

# Датасет
DATASET_NAME = "tatsu-lab/alpaca"  # 52k инструкций
MAX_SEQ_LENGTH = 512

# LoRA параметры
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Обучение
EPOCHS = 1
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4  # эффективный batch = BATCH_SIZE * GRAD_ACCUM_STEPS = 16
LEARNING_RATE = 2e-4
OUTPUT_DIR = "/content/outputs"

# Wandb (опционально — поставьте False чтобы отключить)
USE_WANDB = False

## 4. Загрузка модели с 4-bit квантизацией

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Квантизация — позволяет загрузить большую модель в ограниченную VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Токенизатор
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Модель
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Модель {MODEL_NAME} загружена!")
print(f"Параметров: {model.num_parameters():,}")

## 5. Применение LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Подготовка данных

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, split="train")
print(f"Размер датасета: {len(dataset)} примеров")
print(f"\nПример данных:")
print(dataset[0])

In [ ]:
def format_example(example):
    """Форматирует пример в текстовый промпт."""
    if example.get("input", "").strip():
        text = f"""### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"""
    else:
        text = f"""### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"""
    return text


def tokenize(example):
    text = format_example(example)
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized


tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)
split = tokenized_dataset.train_test_split(test_size=0.05, seed=42)

print(f"Train: {len(split['train'])} примеров")
print(f"Eval:  {len(split['test'])} примеров")

## 7. Обучение

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=200,
    eval_steps=200,
    eval_strategy="steps",
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    report_to="wandb" if USE_WANDB else "none",
    seed=42,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("Начинаем обучение...")
trainer.train()

## 8. Сохранение модели

In [ ]:
# Сохраняем LoRA адаптер
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Модель сохранена в {OUTPUT_DIR}")

# Скачать на локальную машину (опционально)
# from google.colab import files
# !zip -r /content/model.zip {OUTPUT_DIR}
# files.download("/content/model.zip")

## 9. Тестирование модели

In [ ]:
def ask(question, max_new_tokens=256, temperature=0.7):
    """Задать вопрос обученной модели."""
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()


# Тестовые вопросы
questions = [
    "What is machine learning?",
    "Write a Python function to calculate fibonacci numbers.",
    "Explain the difference between AI and ML in simple terms.",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print("-" * 60)

## 10. Gradio Chat UI

Запустите ячейку ниже — откроется интерактивный чат с вашей моделью.
Также получите публичную ссылку, которой можно поделиться.

In [ ]:
import gradio as gr


def chat_fn(message, history, max_tokens, temperature, top_p):
    """Генерация ответа для Gradio чата."""
    prompt = f"### Instruction:\n{message}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=int(max_tokens),
            temperature=float(temperature),
            top_p=float(top_p),
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return response.strip()


with gr.Blocks(title="LLM Chat", theme=gr.themes.Soft()) as demo:
    gr.Markdown(f"# Chat with your Fine-Tuned LLM\n**Model:** `{MODEL_NAME}` | **Adapter:** `{OUTPUT_DIR}`")

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.ChatInterface(
                fn=chat_fn,
                additional_inputs=[
                    gr.Slider(64, 1024, value=256, step=32, label="Max tokens"),
                    gr.Slider(0.1, 2.0, value=0.7, step=0.1, label="Temperature"),
                    gr.Slider(0.1, 1.0, value=0.9, step=0.05, label="Top-p"),
                ],
                examples=[
                    "What is machine learning?",
                    "Write a Python function to sort a list.",
                    "Explain transformers in simple terms.",
                ],
            )

# share=True даёт публичную ссылку (работает ~72 часа)
demo.launch(share=True, debug=True)

## 11. Загрузка на HuggingFace Hub (опционально)

Чтобы сохранить модель в облаке и использовать позже.

In [ ]:
# Раскомментируйте и заполните, чтобы загрузить модель на HuggingFace

# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")  # получите токен на huggingface.co/settings/tokens
#
# HF_USERNAME = "your-username"
# MODEL_REPO = f"{HF_USERNAME}/my-finetuned-llm"
#
# model.push_to_hub(MODEL_REPO)
# tokenizer.push_to_hub(MODEL_REPO)
# print(f"Модель загружена: https://huggingface.co/{MODEL_REPO}")